# Slide 12 classroom demo: semantic, keyword, and hybrid search

Run the cells from top to bottom in VS Code or Jupyter. The default mode runs offline with Python standard library only. It uses **hand-built concept features** to illustrate semantic ranking; these are **not embeddings from a model**. Set `USE_BEDROCK = True` in the configuration cell to use actual Amazon Titan Text Embeddings V2 through Bedrock Runtime. The lexical part uses a small, readable BM25 implementation. The hybrid part combines ranks using reciprocal rank fusion (RRF), not raw BM25 and cosine scores.

**Lesson:** Semantics helps with paraphrases; keywords help with exact error codes, policy identifiers, and names. A hybrid ranking can keep useful candidates from both. This notebook runs an in-memory comparison; it does not provision an OpenSearch index or a Bedrock Knowledge Base.


## 1. Configure the demo

The optional Bedrock mode requires AWS credentials, `boto3`, permission to invoke the model, and availability in your chosen Region. Embedding calls incur charges. Keep the default offline mode when training labs have no AWS access.


In [ ]:
USE_BEDROCK = False   # Offline concept demo by default. Set True to call Bedrock Titan embeddings.
AWS_REGION = "<AWS_REGION>"  # a Region where Titan embeddings are enabled
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
TOP_K = 5
RRF_K = 20  # small value makes rank differences visible in this tiny corpus
SEMANTIC_WEIGHT = 0.55
KEYWORD_WEIGHT = 0.45


## 2. Corpus and class questions

Each document has an ID so you can compare its position across the three result lists. `D1` is a paraphrase match, `D3` contains an exact error code, and `D5` is a plausible but incomplete security document.


In [ ]:
DOCUMENTS = [
    {"id":"D1", "title":"Account recovery runbook", "text":"Employees who cannot sign in can restore access by verifying identity and resetting their credentials. Contact the service desk after three failed attempts."},
    {"id":"D2", "title":"Cloud security overview", "text":"Cloud security best practices include least privilege, encryption, logging, and periodic review of access policies."},
    {"id":"D3", "title":"AccessDeniedException fix", "text":"If an Amazon S3 PutObject request returns AccessDeniedException, check the bucket policy, IAM policy, and KMS key permissions."},
    {"id":"D4", "title":"Account billing", "text":"Customers can update billing details and download invoices from their account settings."},
    {"id":"D5", "title":"Security awareness course", "text":"A course introduces cloud security best practices and explains employee responsibilities at a high level."},
    {"id":"D6", "title":"Policy SEC-204", "text":"Policy SEC-204 requires MFA for administrator access to production AWS accounts."},
    {"id":"D7", "title":"Password management FAQ", "text":"Use strong passwords, keep recovery factors current, and follow the organization's credential rotation rules."},
    {"id":"D8", "title":"Troubleshooting application sign-in", "text":"An authentication failure can occur if the user's identity provider session expired; refresh the session before escalating."},
]
QUERIES = [
    ("Paraphrase", "How can a staff member get back into their account after being locked out?", "D1"),
    ("Exact identifier", "What fixes S3 AccessDeniedException when uploading?", "D3"),
    ("Exact policy", "What does SEC-204 require?", "D6"),
    ("Mixed", "What cloud security best practices cover least privilege?", "D2"),
]
print(f"Corpus: {len(DOCUMENTS)} short documents; queries: {len(QUERIES)}")


## 3. Keyword search with BM25

BM25 rewards matching words and adjusts for document length and term frequency. Tokenization here preserves identifiers such as `SEC-204`. Production OpenSearch analyzers need deliberate configuration for codes and acronyms.


In [ ]:
import math, re
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", text.lower())

TOKENS = [tokenize(d["title"] + " " + d["text"]) for d in DOCUMENTS]
DOC_FREQ = Counter(t for terms in TOKENS for t in set(terms))
AVG_LEN = sum(map(len, TOKENS)) / len(TOKENS)

def bm25(query, k1=1.5, b=0.75):
    terms = set(tokenize(query)); scores = {}
    for doc, words in zip(DOCUMENTS, TOKENS):
        counts = Counter(words); score = 0.0
        for term in terms:
            if term not in counts: continue
            df = DOC_FREQ[term]
            idf = math.log(1 + (len(DOCUMENTS)-df+0.5)/(df+0.5))
            tf = counts[term]
            score += idf * tf * (k1+1) / (tf + k1*(1-b+b*len(words)/AVG_LEN))
        scores[doc["id"]] = score
    return scores


## 4. Semantic scores

Offline mode represents selected *concepts* such as account recovery and object-storage permissions by matching related words. This is a transparent teaching simulation, not a language model. In Bedrock mode, the same ranking step uses real text embeddings and cosine similarity. **Use the same embedding model for queries and documents.**


In [ ]:
CONCEPTS = {
    "account_recovery": {"restore","recover","recovery","locked","lockout","resetting","reset","cannot sign","get back","sign in","login"},
    "permission_error": {"permission","permissions","putobject","bucket policy"},
    "least_privilege": {"least privilege","access policies","access policy","minimal access"},
    "security_practices": {"cloud security","security best practices","encryption","logging","mfa","secure"},
    "billing": {"billing","invoices","invoice"},
    }
def offline_vector(text):
    t=text.lower()
    return [float(sum(1 for phrase in phrases if phrase in t)) for phrases in CONCEPTS.values()]

def cosine(a,b):
    denom = math.sqrt(sum(x*x for x in a))*math.sqrt(sum(y*y for y in b))
    return sum(x*y for x,y in zip(a,b))/denom if denom else 0.0

if USE_BEDROCK:
    import boto3, json
    bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
    def embed(text):
        response=bedrock.invoke_model(
            modelId=EMBEDDING_MODEL_ID,
            body=json.dumps({"inputText": text, "dimensions": 1024, "normalize": True}),
            contentType="application/json", accept="application/json")
        return json.loads(response["body"].read())["embedding"]
else:
    embed=offline_vector

DOC_VECTORS = {d["id"]: embed(d["title"] + ". " + d["text"]) for d in DOCUMENTS}
QUERY_VECTORS = {q: embed(q) for _,q,_ in QUERIES}
def semantic(query):
    return {doc_id:cosine(vec, QUERY_VECTORS[query]) for doc_id,vec in DOC_VECTORS.items()}
print("Semantic mode:", "real Bedrock embeddings" if USE_BEDROCK else "offline concept simulation")


## 5. Fuse ranks and compare results

RRF uses positions rather than adding incompatible BM25 and cosine numbers. Documents with zero BM25 matches do not enter the keyword candidate list. Only documents with positive semantic similarity enter the semantic candidate list; offline concept scores of zero represent no recognized concept.


In [ ]:
def rank(scores, only_positive=False):
    pairs=[(doc_id,score) for doc_id,score in scores.items() if not only_positive or score>0]
    return [doc_id for doc_id,_ in sorted(pairs,key=lambda x:(-x[1],x[0]))]

def hybrid_rrf(semantic_rank, keyword_rank, semantic_weight=SEMANTIC_WEIGHT, keyword_weight=KEYWORD_WEIGHT):
    scores=Counter()
    for weight,items in [(semantic_weight,semantic_rank),(keyword_weight,keyword_rank)]:
        for position,doc_id in enumerate(items,1): scores[doc_id]+=weight/(RRF_K+position)
    return rank(scores)

def show_comparison(label,query,expected):
    ks=bm25(query); ss=semantic(query)
    lexical=rank(ks,only_positive=True)
    vector=rank(ss,only_positive=True)
    fused=hybrid_rrf(vector,lexical)
    title={d["id"]:d["title"] for d in DOCUMENTS}
    print(f"\n{label}: {query}\nExpected evidence: {expected} ({title[expected]})")
    print(f"{'Rank':<5} {'Keyword (BM25)':<28} {'Semantic':<28} {'Hybrid (RRF)':<28}")
    print('-'*94)
    for n in range(TOP_K):
        def name(items): return f"{items[n]} {title[items[n]]}" if n<len(items) else '—'
        print(f"{n+1:<5} {name(lexical):<28} {name(vector):<28} {name(fused):<28}")
    def position(items): return items.index(expected)+1 if expected in items else None
    return {"query":label,"expected":expected,"keyword":position(lexical),"semantic":position(vector),"hybrid":position(fused)}

RESULTS=[show_comparison(*case) for case in QUERIES]
print('\nExpected-document ranks (lower is better; None means not retrieved):')
for row in RESULTS:print(row)


## 6. Discussion and simple tests

1. For the paraphrase, which result appears first in each list? Why might the keyword list miss “restore access” when the query says “get back into account”?
2. For `AccessDeniedException` and `SEC-204`, why should exact identifier handling be tested?
3. Increase `KEYWORD_WEIGHT` to `0.8` and reduce `SEMANTIC_WEIGHT` to `0.2`, rerun cell 5, and note which results move.
4. How would you create a labeled test set and calculate recall@3 or MRR over real queries?
5. Which filters (tenant, classification, freshness) must apply before evidence reaches the answer generator?

**Important:** This demo compares retrieval rankings only. It does not evaluate answer quality, and the offline concept simulation cannot establish production search accuracy. Production OpenSearch can perform lexical, vector, and hybrid search on the same index; test actual corpus, analyzers, permissions, recall, p95 latency, and cost.

AWS references: [Titan embeddings V2 API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-titan-text-embeddings-v2.html), [OpenSearch Serverless neural and hybrid search](https://docs.aws.amazon.com/opensearch-service/latest/developerguide/serverless-configure-neural-search.html).
